In [98]:
# ============================================
# 1. Setup
# ============================================
import requests
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import time

API_KEY = "3bb3e6d99eb16f4ecd4cc4263eddf7d2896fb4df320a984c0c140cb585d9ef4a"   # <-- Replace with your key
BASE_URL = "https://data-api.coindesk.com/index/cc/v1/historical/minutes"

MARKET = "cadli"
INSTRUMENT = "BTC-USD"
LIMIT = 1000       # max allowed per request
AGGREGATE = 1
FILL = True
APPLY_MAPPING = True
FORMAT = "JSON"



In [99]:
# ============================================
# 2. Date Inputs (YOU CAN EDIT THESE)
# ============================================
start_date = "2021-06-01 00:00:00"   # <-- set start
end_date   = "2025-12-01 00:00:00"   # <-- set end

start_ts = int(pd.Timestamp(start_date, tz="UTC").timestamp())
end_ts   = int(pd.Timestamp(end_date,   tz="UTC").timestamp())

print(f"Fetching data from: {start_date}  to  {end_date}")
print(f"Start TS: {start_ts},  End TS: {end_ts}")


1764547200
1764547260

Fetching data from: 2021-06-01 00:00:00  to  2025-12-01 00:00:00
Start TS: 1622505600,  End TS: 1764547200


1764547260

In [100]:
# Test Key
import requests 

response = requests.get('https://data-api.coindesk.com/index/cc/v1/historical/second',
    params={"market":"cadli","instrument":"BTC-USD","limit":1800,"aggregate":1,"fill":"true","apply_mapping":"true","response_format":"JSON","to_ts":1606780800,"api_key":API_KEY},
    headers={"Content-type":"application/json; charset=UTF-8"}
)

json_response = response.json()

print(json_response)

{'Data': {}, 'Err': {'type': 95, 'message': 'Path does not exist.'}}


In [101]:

# ============================================
# 3. Helper: Fetch 1 chunk
# ============================================
def fetch_chunk(to_ts):
    params = {
        "market": MARKET,
        "instrument": INSTRUMENT,
        "limit": LIMIT,
        "aggregate": AGGREGATE,
        "fill": str(FILL).lower(),
        "apply_mapping": str(APPLY_MAPPING).lower(),
        "response_format": FORMAT,
        "to_ts": int(to_ts),
        "api_key": API_KEY
    }

    response = requests.get(
        BASE_URL,
        params=params,
        headers={"Content-type": "application/json; charset=UTF-8"}
    )
    
    if response.status_code != 200:
        print("Error:", response.status_code, response.text)
        return None
    
    return response.json()



In [102]:
# Rough estimate of minutes and chunks needed:
import math

total_minutes = max(1, (end_ts - start_ts) // 60)  # at least 1
n_chunks = math.ceil(total_minutes / LIMIT)


print (f"Total minutes: {total_minutes}")
print(f"Estimated chunks: {n_chunks}")  

Total minutes: 2367360
Estimated chunks: 2368


In [103]:

# ============================================
# 4. Main download loop
# ============================================
current_ts = end_ts
all_rows = []

for _ in tqdm(range(n_chunks)):
    data = fetch_chunk(current_ts)

    if data is None:
        print("Retrying after error...")
        time.sleep(2)
        continue

    # Debug (optional)
    print("Err:", data.get("Err"))
    print("First row:", (data.get("Data") or [None])[0])

    # --- rows are directly in data["Data"] ---
    rows = data.get("Data", [])

    if not isinstance(rows, list) or len(rows) == 0:
        print("No rows in this chunk, stopping.")
        break

    # Append all rows from this chunk
    all_rows.extend(rows)

    # TIMESTAMP is in seconds (not 'time')
    oldest_ts = rows[0]["TIMESTAMP"]   # earliest timestamp in chunk

    # Move 1 minute backward from oldest timestamp
    current_ts = oldest_ts - 60

    if current_ts < start_ts:
        break

    time.sleep(0.1)    # polite pause



  0%|          | 1/2368 [00:01<1:05:19,  1.66s/it]

Err: {}
First row: {'UNIT': 'MINUTE', 'TIMESTAMP': 1764487260, 'TYPE': '267', 'MARKET': 'cadli', 'INSTRUMENT': 'BTC-USD', 'OPEN': 90996.3239220375, 'HIGH': 90996.9143865239, 'LOW': 90985.9752043793, 'CLOSE': 90992.2365354935, 'FIRST_MESSAGE_TIMESTAMP': 1764487260, 'LAST_MESSAGE_TIMESTAMP': 1764487319, 'FIRST_MESSAGE_VALUE': 90996.3239095797, 'HIGH_MESSAGE_VALUE': 90996.9143865239, 'HIGH_MESSAGE_TIMESTAMP': 1764487260, 'LOW_MESSAGE_VALUE': 90985.9752043793, 'LOW_MESSAGE_TIMESTAMP': 1764487291, 'LAST_MESSAGE_VALUE': 90992.2365354935, 'TOTAL_INDEX_UPDATES': 946, 'VOLUME': 32.1625470113189, 'QUOTE_VOLUME': 2926485.27882855, 'VOLUME_TOP_TIER': 18.14826414, 'QUOTE_VOLUME_TOP_TIER': 1651366.08479996, 'VOLUME_DIRECT': 2.31880316, 'QUOTE_VOLUME_DIRECT': 210952.53388282, 'VOLUME_TOP_TIER_DIRECT': 1.97776015000001, 'QUOTE_VOLUME_TOP_TIER_DIRECT': 179934.48225029}


  0%|          | 2/2368 [00:03<1:05:28,  1.66s/it]

Err: {}
First row: {'UNIT': 'MINUTE', 'TIMESTAMP': 1764427260, 'TYPE': '267', 'MARKET': 'cadli', 'INSTRUMENT': 'BTC-USD', 'OPEN': 90717.5043489091, 'HIGH': 90718.2950062242, 'LOW': 90697.439908912, 'CLOSE': 90697.5814118366, 'FIRST_MESSAGE_TIMESTAMP': 1764427260, 'LAST_MESSAGE_TIMESTAMP': 1764427319, 'FIRST_MESSAGE_VALUE': 90717.5188113747, 'HIGH_MESSAGE_VALUE': 90718.2950062242, 'HIGH_MESSAGE_TIMESTAMP': 1764427264, 'LOW_MESSAGE_VALUE': 90697.439908912, 'LOW_MESSAGE_TIMESTAMP': 1764427319, 'LAST_MESSAGE_VALUE': 90697.5814118366, 'TOTAL_INDEX_UPDATES': 1138, 'VOLUME': 57.4854482030408, 'QUOTE_VOLUME': 5213605.35076524, 'VOLUME_TOP_TIER': 29.118260988, 'QUOTE_VOLUME_TOP_TIER': 2641098.86359894, 'VOLUME_DIRECT': 7.76509878, 'QUOTE_VOLUME_DIRECT': 704230.293551004, 'VOLUME_TOP_TIER_DIRECT': 7.19229378, 'QUOTE_VOLUME_TOP_TIER_DIRECT': 652301.749779004}


  0%|          | 3/2368 [00:07<1:42:19,  2.60s/it]

...

Err: {}
First row: {'UNIT': 'MINUTE', 'TIMESTAMP': 1622587260, 'TYPE': '267', 'MARKET': 'cadli', 'INSTRUMENT': 'BTC-USD', 'OPEN': 36460.6655251613, 'HIGH': 36468.5437382216, 'LOW': 36460.6655251613, 'CLOSE': 36468.5437382216, 'FIRST_MESSAGE_TIMESTAMP': 1622587260, 'LAST_MESSAGE_TIMESTAMP': 1622587260, 'FIRST_MESSAGE_VALUE': 36468.5437382216, 'HIGH_MESSAGE_VALUE': 36468.5437382216, 'HIGH_MESSAGE_TIMESTAMP': 1622587260, 'LOW_MESSAGE_VALUE': 36468.5437382216, 'LOW_MESSAGE_TIMESTAMP': 1622587260, 'LAST_MESSAGE_VALUE': 36468.5437382216, 'TOTAL_INDEX_UPDATES': 1, 'VOLUME': 178.53281538043672, 'QUOTE_VOLUME': 6512487.954954959, 'VOLUME_TOP_TIER': 95.51909877465998, 'QUOTE_VOLUME_TOP_TIER': 3480965.4734815857, 'VOLUME_DIRECT': 53.4732978, 'QUOTE_VOLUME_DIRECT': 1947211.4026537645, 'VOLUME_TOP_TIER_DIRECT': 46.75004543, 'QUOTE_VOLUME_TOP_TIER_DIRECT': 1702372.9285241691}


100%|█████████▉| 2367/2368 [1:14:43<00:01,  1.73s/it]

Err: {}
First row: {'UNIT': 'MINUTE', 'TIMESTAMP': 1622527260, 'TYPE': '267', 'MARKET': 'cadli', 'INSTRUMENT': 'BTC-USD', 'OPEN': 37266.4758459546, 'HIGH': 37273.8933057981, 'LOW': 37266.4758459546, 'CLOSE': 37273.8933057981, 'FIRST_MESSAGE_TIMESTAMP': 1622527260, 'LAST_MESSAGE_TIMESTAMP': 1622527260, 'FIRST_MESSAGE_VALUE': 37273.8933057981, 'HIGH_MESSAGE_VALUE': 37273.8933057981, 'HIGH_MESSAGE_TIMESTAMP': 1622527260, 'LOW_MESSAGE_VALUE': 37273.8933057981, 'LOW_MESSAGE_TIMESTAMP': 1622527260, 'LAST_MESSAGE_VALUE': 37273.8933057981, 'TOTAL_INDEX_UPDATES': 1, 'VOLUME': 332.20300146921153, 'QUOTE_VOLUME': 12397646.681024298, 'VOLUME_TOP_TIER': 171.60286647260995, 'QUOTE_VOLUME_TOP_TIER': 6400971.692081434, 'VOLUME_DIRECT': 61.95035102999999, 'QUOTE_VOLUME_DIRECT': 2307134.1103775725, 'VOLUME_TOP_TIER_DIRECT': 41.85921578, 'QUOTE_VOLUME_TOP_TIER_DIRECT': 1559101.576167887}


100%|█████████▉| 2367/2368 [1:14:44<00:01,  1.89s/it]

Err: {}
First row: {'UNIT': 'MINUTE', 'TIMESTAMP': 1622467260, 'TYPE': '267', 'MARKET': 'cadli', 'INSTRUMENT': 'BTC-USD', 'OPEN': 37006.064712418, 'HIGH': 37049.001208701, 'LOW': 37006.064712418, 'CLOSE': 37049.001208701, 'FIRST_MESSAGE_TIMESTAMP': 1622467260, 'LAST_MESSAGE_TIMESTAMP': 1622467260, 'FIRST_MESSAGE_VALUE': 37049.001208701, 'HIGH_MESSAGE_VALUE': 37049.001208701, 'HIGH_MESSAGE_TIMESTAMP': 1622467260, 'LOW_MESSAGE_VALUE': 37049.001208701, 'LOW_MESSAGE_TIMESTAMP': 1622467260, 'LAST_MESSAGE_VALUE': 37049.001208701, 'TOTAL_INDEX_UPDATES': 1, 'VOLUME': 277.06412432692645, 'QUOTE_VOLUME': 10276803.247569678, 'VOLUME_TOP_TIER': 119.9525985147023, 'QUOTE_VOLUME_TOP_TIER': 4439830.635531635, 'VOLUME_DIRECT': 54.92317293999999, 'QUOTE_VOLUME_DIRECT': 2032094.9557138714, 'VOLUME_TOP_TIER_DIRECT': 38.024631389999996, 'QUOTE_VOLUME_TOP_TIER_DIRECT': 1406993.4559580549}


In [104]:
# ============================================
# 5. Convert to DataFrame
# ============================================
df = pd.DataFrame(all_rows)

print(df.columns.tolist())
print(df.head())


['UNIT', 'TIMESTAMP', 'TYPE', 'MARKET', 'INSTRUMENT', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'FIRST_MESSAGE_TIMESTAMP', 'LAST_MESSAGE_TIMESTAMP', 'FIRST_MESSAGE_VALUE', 'HIGH_MESSAGE_VALUE', 'HIGH_MESSAGE_TIMESTAMP', 'LOW_MESSAGE_VALUE', 'LOW_MESSAGE_TIMESTAMP', 'LAST_MESSAGE_VALUE', 'TOTAL_INDEX_UPDATES', 'VOLUME', 'QUOTE_VOLUME', 'VOLUME_TOP_TIER', 'QUOTE_VOLUME_TOP_TIER', 'VOLUME_DIRECT', 'QUOTE_VOLUME_DIRECT', 'VOLUME_TOP_TIER_DIRECT', 'QUOTE_VOLUME_TOP_TIER_DIRECT']
     UNIT   TIMESTAMP TYPE MARKET INSTRUMENT          OPEN          HIGH  \
0  MINUTE  1764487260  267  cadli    BTC-USD  90996.323922  90996.914387   
1  MINUTE  1764487320  267  cadli    BTC-USD  90992.236535  91001.011610   
2  MINUTE  1764487380  267  cadli    BTC-USD  91000.920511  91001.710998   
3  MINUTE  1764487440  267  cadli    BTC-USD  90982.792863  90992.286291   
4  MINUTE  1764487500  267  cadli    BTC-USD  90992.008703  90996.157734   

            LOW         CLOSE  FIRST_MESSAGE_TIMESTAMP  ...  \
0  90985.97

In [105]:
# Remove duplicates by TIMESTAMP
df = df.drop_duplicates(subset=["TIMESTAMP"], keep="last")

# Convert timestamps
df["datetime"] = pd.to_datetime(df["TIMESTAMP"], unit="s", utc=True)

# Keep only date range
df = df[
    (df["datetime"] >= pd.Timestamp(start_date, tz="UTC"))
    & (df["datetime"] <= pd.Timestamp(end_date, tz="UTC"))
]

# Sort chronologically
df = df.sort_values("datetime").reset_index(drop=True)

# Reorder / select columns you actually have
# Map your old logic:
#   open  -> OPEN
#   high  -> HIGH
#   low   -> LOW
#   close -> CLOSE
#   volumefrom -> VOLUME
#   volumeto   -> QUOTE_VOLUME  (or whatever you want)
df = df[
    [
        "datetime",
        "OPEN",
        "HIGH",
        "LOW",
        "CLOSE",
        "VOLUME",
        "QUOTE_VOLUME",
        "VOLUME_TOP_TIER",
        "QUOTE_VOLUME_TOP_TIER",
        "VOLUME_DIRECT",
        "QUOTE_VOLUME_DIRECT",
        "VOLUME_TOP_TIER_DIRECT",
        "QUOTE_VOLUME_TOP_TIER_DIRECT",
    ]
]

df.head(), df.tail()


output_file = f"cadli_btcusd_1m_{start_date[:10]}_to_{end_date[:10]}.csv"
df.to_csv(output_file, index=False)
print("Saved:", output_file)
print("Rows:", df.shape[0])

Saved: cadli_btcusd_1m_2021-06-01_to_2025-12-01.csv
Rows: 2367361
